<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula3-exercicio2-quantas-classes.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 3, exercício 2: Quantas classes

Quantos rótulos diferentes aparecem em `y_treino`? Círculo e quadrado são dois.

O exercício é da aula [3. Poucas fotos](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/03-poucas-fotos.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara o modelo, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q torch torchvision

## Preparação

A preparação desenha 32 fotos de treino e 16 de teste, círculos e quadrados, e treina só a cabeça por duas épocas. A base fica em avaliação.

In [ ]:
def contar(modelo, so_treinavel=False):
    total = 0
    for p in modelo.parameters():
        if so_treinavel and not p.requires_grad:
            continue
        total += p.numel()
    return total


def lote_formas(n, semente, trocar_cor=False):
    """Desenha círculos e quadrados. Classe 0 é círculo, classe 1 é quadrado."""
    g = torch.Generator().manual_seed(semente)
    y = torch.randint(0, 2, (n,), generator=g)
    x = torch.full((n, 3, 64, 64), 0.08)
    yy, xx = torch.meshgrid(torch.arange(64), torch.arange(64), indexing="ij")
    circulo = (yy - 32) ** 2 + (xx - 32) ** 2 <= 16 ** 2
    for i in range(n):
        cor = torch.tensor([0.90, 0.20, 0.15] if int(y[i]) == 0 else [0.15, 0.35, 0.92])
        if trocar_cor:
            cor = cor.flip(0)
        if int(y[i]) == 0:
            x[i, :, circulo] = cor.view(3, 1)
        else:
            x[i, :, 16:48, 16:48] = cor.view(3, 1, 1)
    return x, y


def normalizar(x):
    media = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    desvio = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (x - media) / desvio

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(7)
epocas = 2
x_treino, y_treino = lote_formas(32, semente=11)
x_teste, y_teste = lote_formas(16, semente=29)
x_treino, x_teste = normalizar(x_treino), normalizar(x_teste)

modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in modelo.parameters():
    p.requires_grad = False
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo.eval()
antes_conv = modelo.conv1.weight.detach().clone()
antes_fc = modelo.fc.weight.detach().clone()

criterio = nn.CrossEntropyLoss()
opt = torch.optim.Adam(modelo.fc.parameters(), lr=1e-2)
loader = DataLoader(TensorDataset(x_treino, y_treino), batch_size=8, shuffle=False)
for _ in range(epocas):
    for xb, yb in loader:
        opt.zero_grad()
        perda = criterio(modelo(xb), yb)
        perda.backward()
        opt.step()

mudou_base = (modelo.conv1.weight.detach() - antes_conv).abs().max().item()
mudou_cabeca = (modelo.fc.weight.detach() - antes_fc).abs().max().item()


def acuracia(modelo, x, y):
    modelo.eval()
    with torch.no_grad():
        pred = modelo(x).argmax(1)
    return float((pred == y).float().mean())


acerto = acuracia(modelo, x_teste, y_teste)
print(f"fotos de treino: {len(y_treino)} | classes: {int(y_treino.unique().numel())} | épocas: {epocas}")
print(f"a base andou: {mudou_base:.3e} | a cabeça andou: {mudou_cabeca:.3e}")
print(f"acerto no teste: {acerto:.0%}")

In [ ]:
# @title Correção automática (rode, não precisa ler)
VERIFICAR = {
    1: lambda r: int(r) == len(y_treino),
    2: lambda r: int(r) == int(y_treino.unique().numel()),
    3: lambda r: int(r) == epocas,
    4: lambda r: float(r) < 1e-6,
    5: lambda r: float(r) > 1e-4,
    6: lambda r: abs(float(r) - acuracia(modelo, x_teste, y_teste)) < 1e-5,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Quantos rótulos diferentes aparecem em `y_treino`? Círculo e quadrado são dois.

*Dica:* y_treino.unique() devolve os valores distintos.

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(2, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = int(y_treino.unique().numel())

conferir(2, resposta)

## O que fica

Próximo: [Aula 3, exercício 3: Quantas épocas](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula3-exercicio3-quantas-epocas.ipynb).